# 🚀 Kaggle Nomic Text Embedding Server
This notebook loads the **nomic-ai/nomic-embed-text-v1.5** model and serves it as a FastAPI text embeddings backend. It exposes the API to the public internet using a secure `pyngrok` tunnel.

### ⚙️ Instructions
1. Turn on the **GPU Accelerator** in the notebook settings (select **GPU T4 x2** or **GPU T4 x1**).
2. Add your **Ngrok Auth Token** to Kaggle Secrets:
   - Go to **Add-ons -> Secrets** in the top menu.
   - Add a secret named `NGROK_AUTH_TOKEN` with your token as the value.
   - Make sure the checkbox next to the secret is checked to share it with the notebook.
3. Run all cells in order.

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q \
    transformers \
    accelerate \
    torch \
    fastapi \
    uvicorn \
    pyngrok \
    nest_asyncio \
    pydantic \
    einops

## 🔑 Step 2: Load Secrets and Import Libraries

In [ ]:
import os
import torch
import nest_asyncio
from fastapi import FastAPI
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModel

MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
PORT = 8000

# Fetch secret token from Kaggle user secrets
NGROK_AUTH_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")
    print("✅ Loaded NGROK_AUTH_TOKEN from Kaggle User Secrets.")
except Exception as e:
    print("❌ Failed to load secret from Kaggle User Secrets. Checking environment variables...")
    NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN")

if not NGROK_AUTH_TOKEN:
    print("⚠️ WARNING: NGROK_AUTH_TOKEN is not set. Please add it to Add-ons -> Secrets.")

## 🤖 Step 3: Load Nomic Embedding Model

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print("Loading model...")
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
print(f"✅ Embedding model loaded successfully onto: {model.device}")

## 🛠️ Step 4: Define API Schemas and Router

In [ ]:
app = FastAPI(title="Kaggle Nomic Embedding API")

class EmbedRequest(BaseModel):
    texts: list[str]

class EmbedResponse(BaseModel):
    success: bool
    embeddings: list[list[float]]
    error_message: str

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def get_embeddings_internal(texts: list[str]) -> list[list[float]]:
    import torch.nn.functional as F
    
    encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        model_output = model(**encoded_input)
        
    embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
    embeddings = F.normalize(embeddings, p=2, dim=1)
    
    return embeddings.tolist()

@app.get("/")
def root():
    return {
        "status": "running",
        "model": MODEL_NAME
    }

@app.post("/embed")
def embed(req: EmbedRequest):
    try:
        embeddings = get_embeddings_internal(req.texts)
        return EmbedResponse(
            success=True,
            embeddings=embeddings,
            error_message=""
        )
    except Exception as e:
        return EmbedResponse(
            success=False,
            embeddings=[],
            error_message=str(e)
        )

## 🚀 Step 5: Start Public Tunnel and FastAPI Server

In [ ]:
# Apply nesting patch to allow uvicorn inside Jupyter event loop
nest_asyncio.apply()

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(PORT)
    print("\n" + "="*80)
    print(f"🚀 SERVER PUBLIC URL: {public_url}")
    print("Copy the URL above and paste it into your Backed/.env as AI_SERVER_URL (or embedding server config)")
    print("="*80 + "\n")
else:
    print("\n❌ Unable to set up ngrok tunnel without an authentication token.")

# Launch server directly in the notebook's active event loop
import uvicorn
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()